# Batch ingestion

This notebook loads raw files from the `raw_files` volume into a table. The orders CSV is read from `/Volumes/workspace/data_ingestion/raw_files/orders.csv` and written as a Delta table, both with `CREATE TABLE AS SELECT` and with `COPY INTO`.

Delta keeps table metadata, and file metadata is available when needed through `_metadata`, such as the file name and the time the file was modified.

Databricks can recognize files that were already loaded. A second `COPY INTO` of the same file does not add those rows again, and new data can be merged in when it is needed.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA data_ingestion;

CREATE VOLUME IF NOT EXISTS raw_files;

In [0]:

path = "/Volumes/workspace/data_ingestion/raw_files/orders.csv"

data = """order_id,customer,amount
1,Alice,100
2,Bob,250
3,Anna,150
4,John,300
"""

dbutils.fs.put(path, data, overwrite=True)

Wrote 69 bytes.


True

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/data_ingestion/raw_files/orders.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

order_id,customer,amount,_rescued_data
1,Alice,100,null
2,Bob,250,null
3,Anna,150,null
4,John,300,null


In [0]:
%sql
CREATE OR REPLACE TABLE orders_ctas
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/data_ingestion/raw_files/orders.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM orders_ctas;

order_id,customer,amount,_rescued_data
1,Alice,100,null
2,Bob,250,null
3,Anna,150,null
4,John,300,null


In [0]:
%sql
DESCRIBE DETAIL orders_ctas;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0675da11-70af-4e60-80d4-50033ab8e9c6,workspace.data_ingestion.orders_ctas,null,,2026-10-11T08:08:47.363Z,2026-10-11T08:08:52.000Z,List(),List(),1,1509,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 199ff6ac-6d63-46e4-ae77-f5f8849ae9af)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
CREATE OR REPLACE TABLE orders_copy (
  order_id INT,
  customer STRING,
  amount INT
)
USING DELTA;

In [0]:
%sql
COPY INTO orders_copy
FROM (
    SELECT
        CAST(order_id AS INT) AS order_id,
        CAST(customer AS STRING) AS customer,
        CAST(amount AS INT) AS amount
    FROM '/Volumes/workspace/data_ingestion/raw_files/orders.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
4,4,0


In [0]:
%sql
SELECT * FROM orders_copy;

order_id,customer,amount
1,Alice,100
2,Bob,250
3,Anna,150
4,John,300


In [0]:
%sql
COPY INTO orders_copy
FROM (
    SELECT
        CAST(order_id AS INT) AS order_id,
        CAST(customer AS STRING) AS customer,
        CAST(amount AS INT) AS amount
    FROM '/Volumes/workspace/data_ingestion/raw_files/orders.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

-- check for idempotency

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
%sql
SELECT 'CTAS' AS method, COUNT(*) AS rows
FROM orders_ctas

UNION ALL

SELECT 'COPY INTO' AS method, COUNT(*) AS rows
FROM orders_copy;

method,rows
CTAS,4
COPY INTO,4


We are adding some metadata columns

In [0]:
%sql
ALTER TABLE orders_ctas
ADD COLUMNS (
    source_file STRING,
    file_modified_at TIMESTAMP
);

In [0]:
%sql
INSERT INTO orders_ctas
SELECT
    order_id,
    customer,
    amount,
    _rescued_data,
    _metadata.file_name AS source_file,
    _metadata.file_modification_time AS file_modified_at
FROM read_files(
    '/Volumes/workspace/data_ingestion/raw_files/orders.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
SELECT
    order_id,
    customer,
    source_file,
    file_modified_at
FROM orders_ctas;

order_id,customer,source_file,file_modified_at
1,Alice,orders.csv,2026-10-11T08:08:06.000Z
2,Bob,orders.csv,2026-10-11T08:08:06.000Z
3,Anna,orders.csv,2026-10-11T08:08:06.000Z
4,John,orders.csv,2026-10-11T08:08:06.000Z
1,Alice,null,null
2,Bob,null,null
3,Anna,null,null
4,John,null,null
